# Experiment 1: Logical Self-Consistency Across Related Questions

This notebook tests whether Phi-3-mini-4k-instruct gives **logically consistent** answers across related yes/no questions.

**Core insight:** A model can achieve reasonable per-question *accuracy* while still being logically *inconsistent* — giving answers that contradict each other when asked equivalent questions. This gap between accuracy and consistency reveals that the model is pattern-matching rather than genuinely reasoning.

## Three consistency tests:
1. **Symmetry:** If 'A equals B', does the model agree that 'B equals A'? Same scenario, reversed syntax.
2. **Transitivity:** Given A=B and B=C, is the model consistent across (a) direct ordering, (b) reordered premises, and (c) reversed conclusion?
3. **Reflexivity:** Does the model consistently agree that 'X equals X' for all entities X?

## Key metrics:
- `Accuracy`: % correct answers per question
- `Consistency Score`: % of question-pairs with logically valid joint answers
- `Accuracy-Consistency Gap`: a large positive gap means the model is accidentally correct but logically contradictory

In [ ]:
# ── Setup: clone repo and install dependencies ──────────────────────────────
!git clone https://github.com/Atharvy700/SP-25.git 2>/dev/null || echo 'Already cloned'
%cd SP-25
!pip install transformers accelerate peft -q

In [ ]:
# ── Imports and reproducibility ──────────────────────────────────────────────
import json, random, gc, torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from transformers import AutoTokenizer, AutoModelForCausalLM
from collections import defaultdict

random.seed(42)
np.random.seed(42)

print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ── Generate entity name pool (same method as the repo) ─────────────────────
# Using random 4-7 char uppercase strings prevents the model from
# using world-knowledge shortcuts — it must reason purely from premises.

def gen_names(n=50000, lo=4, hi=7, seed=42):
    rng = random.Random(seed)
    names = set()
    while len(names) < n:
        k = rng.randint(lo, hi)
        names.add(''.join(rng.choices('ABCDEFGHIJKLMNOPQRSTUVWXYZ', k=k)))
    return list(names)

NAMES = gen_names()
print(f'Name pool size: {len(NAMES)}')
print(f'Sample names: {NAMES[:10]}')

In [ ]:
# ── Load Phi-3-mini-4k-instruct ──────────────────────────────────────────────
# We use logit-based yes/no prediction (no generation) for speed and reliability.

if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU: Runtime → Change runtime type → GPU')

gc.collect()
torch.cuda.empty_cache()

MODEL_NAME = 'microsoft/Phi-3-mini-4k-instruct'
MAX_LENGTH  = 384

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Loading model (FP16)...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
)
model.eval()

# Cache yes/no token IDs once
YES_ID = tokenizer('yes', add_special_tokens=False).input_ids[0]
NO_ID  = tokenizer('no',  add_special_tokens=False).input_ids[0]
print(f'yes token id: {YES_ID} | no token id: {NO_ID}')
print('Model ready.')

In [ ]:
# ── Inference helpers ────────────────────────────────────────────────────────

def build_prompt(question_text: str) -> str:
    """Wrap a logical question in the standard prompt format."""
    return (
        'You are a logical reasoning assistant.\n'
        'Answer with exactly one word.\n\n'
        f'{question_text}\n\n'
        'Answer only yes or no:'
    )

@torch.inference_mode()
def predict(prompt: str) -> str:
    """Return 'yes' or 'no' based on logit comparison at the last token."""
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=MAX_LENGTH,
    ).to(model.device)
    logits = model(**inputs).logits[0, -1]          # last token logits
    return 'yes' if logits[YES_ID] > logits[NO_ID] else 'no'

@torch.inference_mode()
def predict_batch(prompts: list, batch_size: int = 8) -> list:
    """Batch inference for speed. Returns list of 'yes'/'no'."""
    results = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        inputs = tokenizer(
            batch,
            return_tensors='pt',
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
        ).to(model.device)
        logits = model(**inputs).logits[:, -1, :]   # [B, vocab]
        for logit in logits:
            results.append('yes' if logit[YES_ID] > logit[NO_ID] else 'no')
    return results

print('Inference helpers defined.')

---
## Test 1: Symmetry Consistency

For each random pair (A, B), we ask two **logically identical** questions:
- **Q_fwd:** `"Suppose A equals B. Does B equal A?"` → correct answer: yes
- **Q_rev:** `"Suppose B equals A. Does A equal B?"` → correct answer: yes

These are the **same logical scenario** with only the surface ordering changed.
A consistent model must give the same answer to both.

**Consistency failure = answering 'yes' to one but 'no' to the other.**

In [ ]:
# ── SYMMETRY CONSISTENCY TEST ────────────────────────────────────────────────
N_SYM_PAIRS = 300
rng = random.Random(42)
name_pool = NAMES[:5000]  # use first 5k names

sym_pairs = [(rng.choice(name_pool), rng.choice(name_pool)) for _ in range(N_SYM_PAIRS)]
# Ensure A != B
sym_pairs = [(A, B) for A, B in sym_pairs if A != B][:N_SYM_PAIRS]

# Build both forward and reverse prompts
q_fwd_prompts = [build_prompt(f'Suppose {A} equals {B}. Does {B} equal {A}?') for A, B in sym_pairs]
q_rev_prompts = [build_prompt(f'Suppose {B} equals {A}. Does {A} equal {B}?') for A, B in sym_pairs]

print(f'Running symmetry test on {len(sym_pairs)} pairs...')
preds_fwd = predict_batch(q_fwd_prompts)
preds_rev = predict_batch(q_rev_prompts)

# Compute metrics
sym_results = []
for (A, B), p_fwd, p_rev in zip(sym_pairs, preds_fwd, preds_rev):
    sym_results.append({
        'A': A, 'B': B,
        'p_fwd': p_fwd, 'p_rev': p_rev,
        'correct_fwd':  p_fwd == 'yes',
        'correct_rev':  p_rev == 'yes',
        'consistent':   p_fwd == p_rev,
        # Logical failure type
        'fail_type': (
            'fwd_only'  if p_fwd == 'yes' and p_rev == 'no'  else
            'rev_only'  if p_fwd == 'no'  and p_rev == 'yes' else
            'both_no'   if p_fwd == 'no'  and p_rev == 'no'  else
            'both_yes'
        )
    })

sym_acc_fwd   = np.mean([r['correct_fwd'] for r in sym_results])
sym_acc_rev   = np.mean([r['correct_rev'] for r in sym_results])
sym_cons      = np.mean([r['consistent']  for r in sym_results])
sym_avg_acc   = (sym_acc_fwd + sym_acc_rev) / 2

# Failure type breakdown
fail_counts = defaultdict(int)
for r in sym_results:
    fail_counts[r['fail_type']] += 1

print()
print('=== SYMMETRY CONSISTENCY ===')
print(f'  Accuracy (forward):  {sym_acc_fwd:.4f}')
print(f'  Accuracy (reverse):  {sym_acc_rev:.4f}')
print(f'  Avg Accuracy:        {sym_avg_acc:.4f}')
print(f'  Consistency Score:   {sym_cons:.4f}')
print(f'  Gap (Acc - Cons):   {sym_avg_acc - sym_cons:+.4f}')
print()
print('  Failure breakdown:')
for k, v in sorted(fail_counts.items()):
    print(f'    {k}: {v} ({100*v/len(sym_results):.1f}%)')

---
## Test 2: Transitivity Consistency

For each random triple (A, B, C), we ask **three related questions** all with the same premises (A=B, B=C):
- **Q_direct:** `"Suppose A equals B. Suppose B equals C. Does A equal C?"` → yes (transitivity)
- **Q_reorder:** `"Suppose B equals C. Suppose A equals B. Does A equal C?"` → yes (same, reordered premises)
- **Q_reverse:** `"Suppose A equals B. Suppose B equals C. Does C equal A?"` → yes (transitivity + symmetry)

A logically consistent model must give the same answer to all three.

In [ ]:
# ── TRANSITIVITY CONSISTENCY TEST ───────────────────────────────────────────
N_TRANS_TRIPLES = 250

# Sample distinct triples
trans_triples = []
used = set()
while len(trans_triples) < N_TRANS_TRIPLES:
    A, B, C = rng.sample(name_pool, 3)
    key = (A, B, C)
    if key not in used:
        used.add(key)
        trans_triples.append((A, B, C))

# Build three question variants
q_direct_prompts  = [
    build_prompt(f'Suppose {A} equals {B}. Suppose {B} equals {C}. Does {A} equal {C}?')
    for A, B, C in trans_triples
]
q_reorder_prompts = [
    build_prompt(f'Suppose {B} equals {C}. Suppose {A} equals {B}. Does {A} equal {C}?')
    for A, B, C in trans_triples
]
q_reverse_prompts = [
    build_prompt(f'Suppose {A} equals {B}. Suppose {B} equals {C}. Does {C} equal {A}?')
    for A, B, C in trans_triples
]

print(f'Running transitivity test on {N_TRANS_TRIPLES} triples (3 question variants each)...')
preds_direct  = predict_batch(q_direct_prompts)
preds_reorder = predict_batch(q_reorder_prompts)
preds_reverse = predict_batch(q_reverse_prompts)

trans_results = []
for (A, B, C), p_d, p_ro, p_rv in zip(trans_triples, preds_direct, preds_reorder, preds_reverse):
    trans_results.append({
        'A': A, 'B': B, 'C': C,
        'p_direct':  p_d,
        'p_reorder': p_ro,
        'p_reverse': p_rv,
        'correct_direct':  p_d  == 'yes',
        'correct_reorder': p_ro == 'yes',
        'correct_reverse': p_rv == 'yes',
        # Pairwise consistency
        'cons_d_ro':  p_d  == p_ro,         # direct vs reordered
        'cons_d_rv':  p_d  == p_rv,         # direct vs reversed
        'all_consistent': p_d == p_ro == p_rv,
        # Logical violations: said yes to direct but no to an equivalent
        'violates_reorder': p_d == 'yes' and p_ro == 'no',
        'violates_reverse': p_d == 'yes' and p_rv == 'no',
    })

acc_direct   = np.mean([r['correct_direct']  for r in trans_results])
acc_reorder  = np.mean([r['correct_reorder'] for r in trans_results])
acc_reverse  = np.mean([r['correct_reverse'] for r in trans_results])
trans_avg_acc = (acc_direct + acc_reorder + acc_reverse) / 3
cons_d_ro    = np.mean([r['cons_d_ro']       for r in trans_results])
cons_d_rv    = np.mean([r['cons_d_rv']       for r in trans_results])
all_cons     = np.mean([r['all_consistent']  for r in trans_results])
viol_reorder = np.mean([r['violates_reorder'] for r in trans_results])
viol_reverse = np.mean([r['violates_reverse'] for r in trans_results])

print()
print('=== TRANSITIVITY CONSISTENCY ===')
print(f'  Accuracy (direct):        {acc_direct:.4f}')
print(f'  Accuracy (reordered):     {acc_reorder:.4f}')
print(f'  Accuracy (reversed):      {acc_reverse:.4f}')
print(f'  Avg Accuracy:             {trans_avg_acc:.4f}')
print()
print(f'  Consistency (direct vs reordered):  {cons_d_ro:.4f}')
print(f'  Consistency (direct vs reversed):   {cons_d_rv:.4f}')
print(f'  All 3 consistent:                   {all_cons:.4f}')
print(f'  Gap (Avg Acc - All Cons):           {trans_avg_acc - all_cons:+.4f}')
print()
print(f'  Violation rate (yes→direct, no→reorder): {viol_reorder:.4f}')
print(f'  Violation rate (yes→direct, no→reverse): {viol_reverse:.4f}')

---
## Test 3: Reflexivity Consistency

For each entity X, we ask: `"Does X equal X?"` — the correct answer is **always yes** by definition.

This is the simplest possible logical test. Any inconsistency here is a serious model failure.

We also test whether the model's answer changes across different entities (it should always say 'yes').

In [ ]:
# ── REFLEXIVITY CONSISTENCY TEST ─────────────────────────────────────────────
N_REFL = 300
refl_entities = rng.sample(name_pool, N_REFL)

# Also construct negative control: ask "Does X equal Y?" with no premise (should lean 'no')
# This validates the model isn't just saying 'yes' to everything
neg_pairs = [(rng.choice(name_pool), rng.choice(name_pool)) for _ in range(N_REFL)]
neg_pairs = [(A, B) for A, B in neg_pairs if A != B][:N_REFL]

refl_prompts = [build_prompt(f'Does {X} equal {X}?') for X in refl_entities]
neg_prompts  = [build_prompt(f'Does {A} equal {B}?') for A, B in neg_pairs]

print(f'Running reflexivity test on {N_REFL} entities...')
refl_preds = predict_batch(refl_prompts)
neg_preds  = predict_batch(neg_prompts)

refl_accuracy     = np.mean([p == 'yes' for p in refl_preds])  # should be 1.0
neg_yes_rate      = np.mean([p == 'yes' for p in neg_preds])   # should be low (no premise)

# Consistency: the model should say 'yes' to ALL reflexive questions
# We measure how stable this is across different entities
refl_results = [{'X': X, 'pred': p, 'correct': p == 'yes'}
                for X, p in zip(refl_entities, refl_preds)]

failed_refl = [r for r in refl_results if not r['correct']]

print()
print('=== REFLEXIVITY CONSISTENCY ===')
print(f'  Accuracy (X equals X → yes): {refl_accuracy:.4f}')
print(f'  Failures (said no):          {len(failed_refl)} / {N_REFL}')
print()
print('  Negative control (A equals B, no premise):')
print(f'  Yes rate: {neg_yes_rate:.4f}  (expected: low)')
print()
if failed_refl:
    print(f'  Sample failures (said no to X=X):')
    for r in failed_refl[:5]:
        print(f'    Does {r["X"]} equal {r["X"]}? → {r["pred"]}')

In [ ]:
# ── VISUALIZATION ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
fig.suptitle(
    'Phi-3-mini-4k-instruct: Accuracy vs. Logical Consistency\n'
    '(Each property tested on ~300 randomly-named entity pairs/triples)',
    fontsize=13, fontweight='bold'
)

ACC_COLOR  = '#2196F3'
CONS_COLOR = '#FF5722'
PERFECT_COLOR = '#4CAF50'

# ── Symmetry panel ──
ax = axes[0]
labels = ['Acc\n(Forward)', 'Acc\n(Reverse)', 'Consistency\nScore']
vals   = [sym_acc_fwd, sym_acc_rev, sym_cons]
colors = [ACC_COLOR, ACC_COLOR, CONS_COLOR]
bars = ax.bar(labels, vals, color=colors, edgecolor='white', linewidth=1.5)
ax.set_ylim(0, 1.15)
ax.set_title('1. Symmetry', fontsize=13, fontweight='bold')
ax.set_ylabel('Score', fontsize=11)
ax.axhline(1.0, color=PERFECT_COLOR, linestyle='--', alpha=0.6, linewidth=1.5)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
# Annotate the gap
ax.annotate('', xy=(2, sym_cons), xytext=(2, sym_avg_acc),
            arrowprops=dict(arrowstyle='<->', color='black', lw=1.5))
ax.text(2.32, (sym_cons + sym_avg_acc)/2,
        f'gap\n{sym_avg_acc - sym_cons:+.3f}', ha='left', va='center', fontsize=9, color='black')
ax.set_ylim(0, 1.2)

# ── Transitivity panel ──
ax = axes[1]
labels = ['Acc\n(Direct)', 'Acc\n(Reorder)', 'Acc\n(Reverse)',
          'Cons\n(D↔R)', 'Cons\n(D↔Rv)', 'All 3\nCons']
vals   = [acc_direct, acc_reorder, acc_reverse, cons_d_ro, cons_d_rv, all_cons]
colors = [ACC_COLOR]*3 + [CONS_COLOR]*3
bars = ax.bar(labels, vals, color=colors, edgecolor='white', linewidth=1.5)
ax.set_ylim(0, 1.2)
ax.set_title('2. Transitivity', fontsize=13, fontweight='bold')
ax.set_ylabel('Score', fontsize=11)
ax.axhline(1.0, color=PERFECT_COLOR, linestyle='--', alpha=0.6, linewidth=1.5)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# ── Reflexivity panel ──
ax = axes[2]
labels = ['Accuracy\n(X=X → yes)', 'Neg. Control\n(A=B, no premise)']
vals   = [refl_accuracy, neg_yes_rate]
colors = [ACC_COLOR, '#9E9E9E']
bars = ax.bar(labels, vals, color=colors, edgecolor='white', linewidth=1.5)
ax.set_ylim(0, 1.15)
ax.set_title('3. Reflexivity', fontsize=13, fontweight='bold')
ax.set_ylabel('Score', fontsize=11)
ax.axhline(1.0, color=PERFECT_COLOR, linestyle='--', alpha=0.6, linewidth=1.5)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Legend
acc_patch  = mpatches.Patch(color=ACC_COLOR,   label='Accuracy (per-question)')
cons_patch = mpatches.Patch(color=CONS_COLOR,  label='Consistency (joint logic)')
perf_line  = plt.Line2D([0], [0], color=PERFECT_COLOR, linestyle='--', label='Perfect (1.0)')
fig.legend(handles=[acc_patch, cons_patch, perf_line],
           loc='lower center', ncol=3, fontsize=10, bbox_to_anchor=(0.5, -0.05))

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig('experiment1_consistency_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: experiment1_consistency_results.png')

In [ ]:
# ── FINAL SUMMARY TABLE ──────────────────────────────────────────────────────
print('=' * 60)
print('SUMMARY: ACCURACY vs. LOGICAL CONSISTENCY — Phi-3-mini-4k')
print('=' * 60)
print(f'{"Property":<16} {"Avg Accuracy":>14} {"Consistency":>13} {"Gap":>10}')
print('-' * 60)

rows = [
    ('Symmetry',     sym_avg_acc,   sym_cons),
    ('Transitivity', trans_avg_acc, all_cons),
    ('Reflexivity',  refl_accuracy, refl_accuracy),  # consistency = accuracy for reflexivity
]

for name, acc, cons in rows:
    gap = acc - cons
    print(f'{name:<16} {acc:>14.4f} {cons:>13.4f} {gap:>+10.4f}')

print('=' * 60)
print()
print('Interpretation:')
print('  Positive gap → model answers correctly PER QUESTION but gives')
print('  logically contradictory answers across related questions.')
print()
print('  This exposes a critical failure mode: the model is pattern-matching')
print('  surface features of individual prompts, NOT applying logical rules.')
print()

# Additional: show violation examples
print('─' * 60)
print('SAMPLE VIOLATIONS (model gave inconsistent symmetric answers):')
print('─' * 60)
viol = [r for r in sym_results if not r['consistent']][:5]
for r in viol:
    print(f"  Q_fwd: 'Suppose {r['A']} equals {r['B']}. Does {r['B']} equal {r['A']}?' → {r['p_fwd']}")
    print(f"  Q_rev: 'Suppose {r['B']} equals {r['A']}. Does {r['A']} equal {r['B']}?' → {r['p_rev']}")
    print()

print('─' * 60)
print('SAMPLE VIOLATIONS (transitivity inconsistency — direct vs reordered):')
print('─' * 60)
viol_t = [r for r in trans_results if not r['cons_d_ro']][:3]
for r in viol_t:
    print(f"  Q_direct:  'A={r['A']}, B={r['B']}, C={r['C']}. Does A=C?' → {r['p_direct']}")
    print(f"  Q_reorder: 'B=C first, then A=B. Does A=C?'                → {r['p_reorder']}")
    print()

In [ ]:
# ── SAVE RESULTS TO JSON ─────────────────────────────────────────────────────
results_summary = {
    'model': MODEL_NAME,
    'symmetry': {
        'n_pairs':       len(sym_results),
        'accuracy_fwd':  sym_acc_fwd,
        'accuracy_rev':  sym_acc_rev,
        'avg_accuracy':  sym_avg_acc,
        'consistency':   sym_cons,
        'gap':           sym_avg_acc - sym_cons,
        'failure_breakdown': dict(fail_counts),
    },
    'transitivity': {
        'n_triples':        len(trans_results),
        'accuracy_direct':  acc_direct,
        'accuracy_reorder': acc_reorder,
        'accuracy_reverse': acc_reverse,
        'avg_accuracy':     trans_avg_acc,
        'consistency_direct_reorder': cons_d_ro,
        'consistency_direct_reverse': cons_d_rv,
        'all_consistent':   all_cons,
        'gap':              trans_avg_acc - all_cons,
        'violation_rate_reorder': viol_reorder,
        'violation_rate_reverse': viol_reverse,
    },
    'reflexivity': {
        'n_entities':         N_REFL,
        'accuracy':           refl_accuracy,
        'neg_control_yes_rate': neg_yes_rate,
        'n_failures':         len(failed_refl),
    },
}

with open('experiment1_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print('Results saved to experiment1_results.json')
print(json.dumps(results_summary, indent=2))